# Initialization

In [93]:
import pandas as pd
epoch_in_minutes = 5
solver_max_timeout_in_seconds = 60
selected_date = pd.to_datetime("2022-06-01").date()
location = "California"

In [ ]:
import random
def generate_jobs_info(num_jobs):
    short_period, medium_period, long_period = 1, 5, 50 # processing duration in epochs
    jobs_pool = [  # task = (machine_id, processing_epoch_numbers)
        [  # Job 0
            [(0, short_period), (1, short_period), (2, short_period)],  # task 0
            [(0, short_period), (1, short_period), (2, short_period)],  # task 1
        ],
        [  # Job 1
            [(0, short_period), (1, short_period), (2, short_period)],  # task 0
            [(0, medium_period), (1, medium_period), (2, medium_period)],  # task 1
        ],
        [  # Job 2
            [(0, short_period), (1, short_period), (2, short_period)],  # task 0
            [(0, long_period), (1, long_period), (2, long_period)],  # task 1
        ],
        [  # Job 3
            [(0, medium_period), (1, medium_period), (2, medium_period)],  # task 0
            [(0, short_period), (1, short_period), (2, short_period)],  # task 1
        ],
        [  # Job 4
            [(0, medium_period), (1, medium_period), (2, medium_period)],  # task 0
            [(0, medium_period), (1, medium_period), (2, medium_period)],  # task 1
        ],
        [  # Job 5
            [(0, medium_period), (1, medium_period), (2, medium_period)],  # task 0
            [(0, long_period), (1, long_period), (2, long_period)],  # task 1
        ],
        [  # Job 6
            [(0, long_period), (1, long_period), (2, long_period)],  # task 0
            [(0, short_period), (1, short_period), (2, short_period)],  # task 1
        ],
        [  # Job 7
            [(0, long_period), (1, long_period), (2, long_period)],  # task 0
            [(0, medium_period), (1, medium_period), (2, medium_period)],  # task 1
        ],
        [  # Job 8
            [(0, long_period), (1, long_period), (2, long_period)],  # task 0
            [(0, long_period), (1, long_period), (2, long_period)],  # task 1
        ],
    ]
    jobs_data = [jobs_pool[0], jobs_pool[4], jobs_pool[8]] #stats.txt
    # jobs_data = [jobs_pool[8], jobs_pool[4], jobs_pool[4]] #stats2.txt
    return jobs_data

In [ ]:
# jobs_data = [  # task = (machine_id, processing_epoch_numbers)
#     [  # Job 0
#         [(0, 3), (1, 1), (2, 5)],  # task 0
#         [(0, 2), (1, 4), (2, 6)],  # task 1
#         [(0, 2), (1, 3), (2, 1)],  # task 2
#     ],
#     [  # Job 1
#         [(0, 2), (1, 3), (2, 4)],
#         [(0, 1), (1, 5), (2, 4)],
#         [(0, 2), (1, 1), (2, 4)],
#     ],
#     [  # Job 2
#         [(0, 2), (1, 1), (2, 4)],
#         [(0, 2), (1, 3), (2, 4)],
#         [(0, 3), (1, 1), (2, 5)],
#     ],
# ]
# jobs_arrival_epoch = [0, 1, 2]

jobs_data = generate_jobs_info(num_jobs=3)
num_jobs = len(jobs_data)


In [75]:
jobs_arrival_epoch = sorted([random.randrange(0, 24 * 60 // epoch_in_minutes) for _ in range(num_jobs)])
jobs_arrival_epoch

[10, 185, 267]

In [ ]:
jobs_arrival_epoch = [10, 185, 267]

In [88]:
# Server Info
num_machines = 3
power = 1 # machines are homogeneous

# Horizon in epochs
# horizon = 0
# for job_index, job in enumerate(jobs_data):
#     for task in job:
#         max_task_duration = 0
#         for alternative in task:
#             max_task_duration = max(max_task_duration, alternative[1] + jobs_arrival_epoch[job_index])
#         horizon += max_task_duration
        
initial_horizon = 2 * 24 * 60 // epoch_in_minutes # 2days - initial horizon - will change based on the minimum makespan

## Carbon Intensity

In [ ]:
# Example CI
CI = [10, 20, 15, 25, 5, 10, 20, 30, 25, 15, 10, 5, 5, 10, 20,
      10, 20, 15, 25, 5, 10, 20, 30, 25, 15, 10, 5, 5, 10, 20,
      10, 20, 15, 25, 5, 10, 20, 30, 25, 15, 10, 5, 5, 10, 20,
      10, 20, 15, 25, 5, 10, 20, 30, 25, 15, 10, 5, 5, 10, 20]  # length must be ≥ horizon

In [77]:
import pandas as pd
carbon_trace = {}
carbon_trace_spec_day = {}
carbon_trace_spec_day_per_epoch = {}
carbon_trace[location] = pd.read_csv(f"../CarbonTrace/US-CAL-CISO.csv")[['datetime', 'carbon_intensity_avg']]
carbon_trace[location]['datetime'] = pd.to_datetime(carbon_trace["California"]['datetime'])

In [117]:
import pandas as pd
next_date = selected_date + pd.Timedelta(days=1)
date_windows = [selected_date]
for extra_day in range(7): # one week
    next_date = selected_date + pd.Timedelta(days=extra_day)
    date_windows.append(next_date)
mask = carbon_trace[location]["datetime"].dt.date.isin(date_windows)
carbon_trace_spec_day[location] = carbon_trace[location][mask].copy()
carbon_trace_spec_day[location]["Hour"] = carbon_trace[location]["datetime"].dt.hour

In [118]:
print(len(carbon_trace_spec_day[location]))
carbon_trace_spec_day[location].head(3)

168


,datetime,carbon_intensity_avg,Hour
21168,2022-06-01 00:00:00+00:00,164.58,0
21169,2022-06-01 01:00:00+00:00,207.90,1
21170,2022-06-01 02:00:00+00:00,262.44,2


In [119]:
carbon_trace_spec_day_per_epoch[location] = []
epoch_num_in_one_hour = 60 // epoch_in_minutes
intensities = carbon_trace_spec_day[location]['carbon_intensity_avg'].tolist()
for intensity in intensities:
    for i in range(epoch_num_in_one_hour):
        carbon_trace_spec_day_per_epoch[location].append(int(round(intensity)))

In [120]:
carbon_trace_spec_day_per_epoch[location][:5]

[165, 165, 165, 165, 165]

In [121]:
len(carbon_trace_spec_day_per_epoch[location])

2016

# Makespan minimization

In [92]:
from ortools.sat.python import cp_model

def flexible_jobshop():
    print(f"Arrival epochs = {jobs_arrival_epoch}")
    model = cp_model.CpModel()

    all_tasks = {}
    all_machines = [[] for _ in range(num_machines)]
    # Create variables
    for job_id, job in enumerate(jobs_data):
        for task_id, alternatives in enumerate(job):
            for m_id, duration in alternatives:
                suffix = f'_{job_id}_{task_id}_{m_id}'
                start = model.NewIntVar(0, initial_horizon, 'start' + suffix)
                end = model.NewIntVar(0, initial_horizon, 'end' + suffix)
                presence = model.NewBoolVar('presence' + suffix)
                interval = model.NewOptionalIntervalVar(start, duration, end, presence, 'interval' + suffix)

                all_tasks[(job_id, task_id, m_id)] = (start, end, interval, presence)
                all_machines[m_id].append((start, duration, interval))

    # Add constraints
    for job_id, job in enumerate(jobs_data):
        for task_id, alternatives in enumerate(job):
            # Only one machine must be selected for each operation
            presences = [all_tasks[(job_id, task_id, m_id)][3] for m_id, _ in alternatives]
            model.AddExactlyOne(presences)

            # Precedence constraints between operations
            if task_id > 0:
                prev_alts = jobs_data[job_id][task_id - 1]
                curr_alts = alternatives
                for prev_m_id, _ in prev_alts:
                    for curr_m_id, _ in curr_alts:
                        prev_end = all_tasks[(job_id, task_id - 1, prev_m_id)][1]
                        curr_start = all_tasks[(job_id, task_id, curr_m_id)][0]
                        prev_presence = all_tasks[(job_id, task_id - 1, prev_m_id)][3]
                        curr_presence = all_tasks[(job_id, task_id, curr_m_id)][3]
                        model.Add(curr_start >= prev_end).OnlyEnforceIf([prev_presence, curr_presence])
            # first task's start time must be after arrival time
            if task_id == 0:
                curr_alts = alternatives
                for curr_m_id, _ in curr_alts:
                    curr_presence = all_tasks[(job_id, task_id, curr_m_id)][3]
                    curr_start = all_tasks[(job_id, task_id, curr_m_id)][0]
                    model.Add(curr_start >= jobs_arrival_epoch[job_id]).OnlyEnforceIf(curr_presence)
            
            

    # Machine capacity constraints: no overlapping intervals
    for machine_id in range(num_machines):
        machine_tasks = all_machines[machine_id]
        model.AddNoOverlap([interval for _, _, interval in machine_tasks]) # For every pair of intervals in the list, the solver ensures that if both are present, then their execution windows do not overlap

    # Objective: minimize makespan (latest end time)
    all_ends = [end for (_, end, _, _) in all_tasks.values()]
    makespan = model.NewIntVar(0, initial_horizon, 'makespan')
    model.AddMaxEquality(makespan, all_ends)
    model.Minimize(makespan)

    # Solve model
    solver = cp_model.CpSolver()
    status = solver.Solve(model)

    # Output solution
    if status in [cp_model.OPTIMAL, cp_model.FEASIBLE]:
        print(f'Minimized makespan: {solver.Value(makespan)}')
        for (j, t, m), (start, end, interval, presence) in all_tasks.items():
            if solver.Value(presence):
                print(f'Job {j}, Task {t} on Machine {m} → Start: {solver.Value(start)}, End: {solver.Value(end)}')
        return solver.Value(makespan)
    else:
        print("No solution found.")
        return None

# Run the example
global_minimum_makespan = flexible_jobshop()

Arrival epochs = [10, 185, 267]
Minimized makespan: 367
Job 0, Task 0 on Machine 2 → Start: 10, End: 11
Job 0, Task 1 on Machine 1 → Start: 11, End: 12
Job 1, Task 0 on Machine 0 → Start: 185, End: 190
Job 1, Task 1 on Machine 0 → Start: 190, End: 195
Job 2, Task 0 on Machine 1 → Start: 267, End: 317
Job 2, Task 1 on Machine 0 → Start: 317, End: 367


In [91]:
jobs_arrival_epoch

[10, 185, 267]

# Carbon consumption minimization

In [130]:
import math
slack_coeff = 1.2
max_allowed_makespan = int(math.ceil(slack_coeff * global_minimum_makespan)) # in epoch
makespan_activated = False

In [131]:
from ortools.sat.python import cp_model

def flexible_jobshop():
    model = cp_model.CpModel()
    if makespan_activated:
        horizon = max_allowed_makespan
        print(f"horizon = {horizon} -> {slack_coeff} x minimum makespan")
    else:
        horizon = initial_horizon
        print(f"horizon = {horizon} - No limit!")
    print(f"Arrival epochs = {jobs_arrival_epoch}")
    all_tasks = {}
    all_machines = [[] for _ in range(num_machines)]

    # Create variables
    for job_id, job in enumerate(jobs_data):
        for task_id, alternatives in enumerate(job):
            for m_id, duration in alternatives:
                suffix = f'_{job_id}_{task_id}_{m_id}'
                start = model.NewIntVar(0, horizon, 'start' + suffix)
                end = model.NewIntVar(0, horizon, 'end' + suffix)
                presence = model.NewBoolVar('presence' + suffix)
                interval = model.NewOptionalIntervalVar(start, duration, end, presence, 'interval' + suffix)

                all_tasks[(job_id, task_id, m_id)] = (start, end, interval, presence)
                # all_machines[m_id].append((start, duration, interval))
                all_machines[m_id].append((start, duration, interval, presence))

    # Add constraints
    for job_id, job in enumerate(jobs_data):
        for task_id, alternatives in enumerate(job):
            # Only one machine must be selected for each operation
            presences = [all_tasks[(job_id, task_id, m_id)][3] for m_id, _ in alternatives]
            model.AddExactlyOne(presences)

            # Precedence constraints between operations
            if task_id > 0:
                prev_alts = jobs_data[job_id][task_id - 1]
                curr_alts = alternatives
                for prev_m_id, _ in prev_alts:
                    for curr_m_id, _ in curr_alts:
                        prev_end = all_tasks[(job_id, task_id - 1, prev_m_id)][1]
                        curr_start = all_tasks[(job_id, task_id, curr_m_id)][0]
                        prev_presence = all_tasks[(job_id, task_id - 1, prev_m_id)][3]
                        curr_presence = all_tasks[(job_id, task_id, curr_m_id)][3]
                        model.Add(curr_start >= prev_end).OnlyEnforceIf([prev_presence, curr_presence])
            # first task's start time must be after arrival epoch
            if task_id == 0:
                curr_alts = alternatives
                for curr_m_id, _ in curr_alts:
                    curr_presence = all_tasks[(job_id, task_id, curr_m_id)][3]
                    curr_start = all_tasks[(job_id, task_id, curr_m_id)][0]
                    model.Add(curr_start >= jobs_arrival_epoch[job_id]).OnlyEnforceIf(curr_presence)
            
            

    # Machine capacity constraints: no overlapping intervals
    for machine_id in range(num_machines):
        machine_tasks = all_machines[machine_id]
        model.AddNoOverlap([interval for _, _, interval, _ in machine_tasks]) # For every pair of intervals in the list, the solver ensures that if both are present, then their execution windows do not overlap

    # Limiting makespan by an upper-bound
    if makespan_activated:
        all_ends = [end for (_, end, _, _) in all_tasks.values()]
        makespan = model.NewIntVar(0, horizon, 'makespan')
        model.AddMaxEquality(makespan, all_ends)
        model.Add(makespan <= max_allowed_makespan)
    
    
    # Carbon-aware objective
    total_carbon_terms = []
    active_vars = []
    b_vars = []
    for t in range(horizon):
        active_vars.append([])
        for m in range(num_machines):
            active = model.NewBoolVar(f'active_{t}_{m}')
            active_vars[t].append(active)
            relevant_presences = []
            for (start, dur, interval, presence) in all_machines[m]:
                b = model.NewBoolVar(f'running_{t}_{m}') # b[t][m] = True when a task is running on machine m and the time t is within the [start, end] of the task                
                in_window1 = model.NewBoolVar('')
                in_window2 = model.NewBoolVar('')
                model.Add(start <= t).OnlyEnforceIf(in_window1)
                model.Add(start > t).OnlyEnforceIf(in_window1.Not())
                model.Add(t < start + dur).OnlyEnforceIf(in_window2)
                model.Add(t >= start + dur).OnlyEnforceIf(in_window2.Not())
                model.AddImplication(b, in_window1)
                model.AddImplication(b, in_window2)
                model.AddImplication(b, presence)
                model.AddBoolOr(in_window1.Not(), in_window2.Not(), presence.Not(), b)
                relevant_presences.append(b)
            b_vars.append(relevant_presences)
            
            model.AddMaxEquality(active, relevant_presences)

            # Compute carbon emission at this time step
            # carbon_term = model.NewIntVar(0, carbon_trace_spec_day_per_epoch[location][t] * power[m], f'carbon_{t}_{m}')
            # model.AddMultiplicationEquality(carbon_term, [active, carbon_trace_spec_day_per_epoch[location][t] * power[m]])
            carbon_value = carbon_trace_spec_day_per_epoch[location][t] * power
            carbon_term = model.NewIntVar(0, carbon_value, f'carbon_{t}_{m}')
            model.Add(carbon_term == carbon_value).OnlyEnforceIf(active)
            model.Add(carbon_term == 0).OnlyEnforceIf(active.Not())
            total_carbon_terms.append(carbon_term)

    total_carbon = model.NewIntVar(0, 1000000, 'total_carbon')
    model.Add(total_carbon == sum(total_carbon_terms))
    model.Minimize(total_carbon)
    # model.Minimize(makespan)

    # Solve model
    solver = cp_model.CpSolver()
    solver.parameters.max_time_in_seconds = solver_max_timeout_in_seconds
    status = solver.Solve(model)

    # Output solution
    if status in [cp_model.OPTIMAL, cp_model.FEASIBLE]:
        total_carbon_consumption = solver.Value(total_carbon) * (epoch_in_minutes / 60)
        if makespan_activated:
            print(f"✅ Total carbon emitted: {total_carbon_consumption:.2f} Kg with maximum allowed makespan = {max_allowed_makespan}")
        else:
            print(f"✅ Total carbon emitted: {total_carbon_consumption:.2f} Kg with 2 days limit")
        if status == cp_model.OPTIMAL:
            print(f"✅ Solution is also optimal!")
        for (j, t, m), (start, end, _, presence) in all_tasks.items():
            if solver.Value(presence):
                print(f'Job {j}, Task {t} on Machine {m} → Start: {solver.Value(start)}, End: {solver.Value(end)}')
    elif status == cp_model.INFEASIBLE:
        print("❌ Problem is infeasible.")
    else:
        print("❓ Solver stopped (possibly due to timeout) with status:", status)

# Run the example
flexible_jobshop()

horizon = 576 - No limit!
Arrival epochs = [10, 185, 267]
✅ Total carbon emitted: 1380.33 Kg with 2 days limit
Job 0, Task 0 on Machine 2 → Start: 204, End: 205
Job 0, Task 1 on Machine 2 → Start: 205, End: 206
Job 1, Task 0 on Machine 1 → Start: 204, End: 209
Job 1, Task 1 on Machine 0 → Start: 209, End: 214
Job 2, Task 0 on Machine 2 → Start: 468, End: 518
Job 2, Task 1 on Machine 2 → Start: 518, End: 568


In [114]:
len(carbon_trace_spec_day_per_epoch[location])

576

In [106]:
consumption = (
    carbon_trace_spec_day_per_epoch[location][204] +
    carbon_trace_spec_day_per_epoch[location][205] +
    sum(carbon_trace_spec_day_per_epoch[location][204:209]) +
    sum(carbon_trace_spec_day_per_epoch[location][209:214]) +
    sum(carbon_trace_spec_day_per_epoch[location][267:317]) +
    sum(carbon_trace_spec_day_per_epoch[location][317:367])
)

In [109]:
carbon_trace_spec_day_per_epoch[location][204:209]

[134, 134, 134, 134, 134]